### Exercise 1: Image Classification with a Pre-Trained Model

Load a pre-trained model (e.g., ResNet18 from torchvision), pass an image through it, and print the top-5 predicted classes with confidence scores. Swap out the final classification head to classify a small custom dataset (e.g., 3 categories of your choice) and fine-tune the model on it.

**Skills practiced:** Transfer learning, model loading, basic PyTorch workflow, custom datasets.

In [6]:
import torch                                # Core deep learning library, includes tensors, autogradient, etc.
from torchvision import models, transforms  # CV library, contains pretrained models and image transform functions
from PIL import Image                       # Used for loading images into memory
import json                                 # Used to decode classification labels
import urllib.request                       # Used to decode classification labels

In [7]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # Load the specific model into memory
model.eval()                                                     # Switches the model out of training mode and into inference mode

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\laesc/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:01<00:00, 39.1MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

**Neural networks need an image in a tensor of a specific size, scaled and normalized the same as the data it was trained on**

What I should know:
- The shape of the pipeline
  - resize -> crop -> tensor -> normalize
- Why normalization matters
- Why the crop size is 224
- That the mean and std should match the training data

What is acceptable to look up:
- The literal mean/std triplet values
- Exact resize/crop dimensions per model architecture
- Exact API syntax

In [8]:
# Preprocess the image for the model manually
# preprocess = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])

# Auto matches the training preprocessing
weights = models.ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

# Apply preprocessing to the image
img = Image.open("1_data/kittens/kitten1.jpg").convert("RGB")
input_tensor = preprocess(img)

PyTorch models expect a batch of inputs as such: (batch_size, channels, height, width)

After preprocessing, a single image has 3-dimensions, but the model expects 4: 

`input_tensor.shape  # torch.Size([3, 224, 224])`

Whenever feeding a tensor into a model, documentation (or error messages) will supply expected dimensions. `squeeze`/`unsqueeze`/`view`/`reshape` are the tools to fix this. Printing shape constantly helps with this:

```
print(input_tensor.shape)   # torch.Size([3, 224, 224])
print(input_batch.shape)    # torch.Size([1, 3, 224, 224])
print(output.shape)         # torch.Size([1, 1000])
```

In [9]:
# Add a batch dimension
input_batch = input_tensor.unsqueeze(0) 

## Run the image through the model and get top 5 predictions

In [13]:
# no_grad disables gradient tracking
with torch.no_grad(): 
    # Outputs raw logits, need to transform to probabilities with softmax
    output = model(input_batch)

# Transform logits into probability distribution
probabilities = torch.nn.functional.softmax(output[0], dim=0)

print(probabilities)

# Get top 5 probabilities
top_5_prob, top_5_idx = torch.topk(probabilities, 5)

tensor([1.2408e-05, 3.8106e-06, 1.4793e-07, 1.8990e-07, 1.2227e-06, 9.7020e-08,
        3.2581e-07, 3.5573e-06, 6.7058e-06, 2.2679e-05, 4.3267e-06, 1.1133e-05,
        5.4406e-06, 1.7940e-06, 2.8195e-06, 1.7644e-04, 1.6931e-05, 7.4407e-05,
        1.0678e-05, 6.6742e-07, 1.1439e-05, 3.2655e-04, 1.4861e-05, 6.2405e-06,
        1.0278e-04, 3.6203e-06, 1.3599e-05, 9.7785e-06, 1.0447e-06, 9.1798e-07,
        4.8012e-05, 1.8194e-06, 1.4863e-06, 2.1110e-08, 6.8708e-08, 6.3216e-06,
        2.3802e-06, 5.4031e-06, 4.5905e-06, 3.8995e-06, 2.2735e-06, 3.1865e-06,
        7.2041e-07, 6.5835e-06, 2.2747e-06, 1.5720e-06, 7.4468e-06, 3.0421e-06,
        8.1635e-06, 6.8527e-07, 1.1178e-06, 1.5822e-06, 2.7649e-06, 7.9998e-07,
        7.6487e-07, 2.7416e-06, 3.9463e-07, 1.7436e-06, 7.1768e-06, 1.5925e-05,
        1.5966e-06, 7.8785e-07, 2.3172e-06, 2.9909e-06, 6.1653e-06, 3.0571e-07,
        3.5915e-06, 7.3105e-06, 9.5928e-07, 6.4668e-08, 2.4795e-06, 3.0840e-07,
        2.6180e-05, 2.7273e-06, 1.5708e-

### Get class names from model documentation

In [23]:
url = 'https://gist.githubusercontent.com/ageitgey/4e1342c10a71981d0b491e1b8227328b/raw/24d78ea09a31fdff540a8494886e0051e3ad68f8/imagenet_classes.txt'
categories = urllib.request.urlopen(url).read().decode("utf-8").split("\n")
categories = categories[4:]
categories = [line.split(", ", 1)[1] for line in categories if line.strip()]
print(categories)


['tench', 'goldfish', 'great_white_shark', 'tiger_shark', 'hammerhead', 'electric_ray', 'stingray', 'cock', 'hen', 'ostrich', 'brambling', 'goldfinch', 'house_finch', 'junco', 'indigo_bunting', 'robin', 'bulbul', 'jay', 'magpie', 'chickadee', 'water_ouzel', 'kite', 'bald_eagle', 'vulture', 'great_grey_owl', 'European_fire_salamander', 'common_newt', 'eft', 'spotted_salamander', 'axolotl', 'bullfrog', 'tree_frog', 'tailed_frog', 'loggerhead', 'leatherback_turtle', 'mud_turtle', 'terrapin', 'box_turtle', 'banded_gecko', 'common_iguana', 'American_chameleon', 'whiptail', 'agama', 'frilled_lizard', 'alligator_lizard', 'Gila_monster', 'green_lizard', 'African_chameleon', 'Komodo_dragon', 'African_crocodile', 'American_alligator', 'triceratops', 'thunder_snake', 'ringneck_snake', 'hognose_snake', 'green_snake', 'king_snake', 'garter_snake', 'water_snake', 'vine_snake', 'night_snake', 'boa_constrictor', 'rock_python', 'Indian_cobra', 'green_mamba', 'sea_snake', 'horned_viper', 'diamondback', 

### Print the Output

In [25]:
for prob, idx in zip(top_5_prob, top_5_idx):
    print(f"{categories[idx]}: {prob.item()*100:.2f}%")

tiger_cat: 58.01%
lynx: 30.75%
tabby: 6.97%
Egyptian_cat: 1.56%
tiger: 0.27%
